# Lane Detection with HOG + Logistic Regression on CULane

This notebook trains a patch-based binary classifier to detect lane pixels in dashcam footage using the CULane Dataset.

In [ ]:
import numpy as np
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.linear_model import LogisticRegression
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, jaccard_score

# 1. Configuration

Dataset root and split list paths for the CULane Kaggle mount.

In [ ]:
#Dataset paths (CULane on Kaggle)
#DATA_ROOT points to the mounted dataset; LIST_ROOT contains the split text files.
DATA_ROOT  = '/kaggle/input/datasets/manideep1108/culane/CULane/'
LIST_ROOT  = DATA_ROOT + 'list/list/'

TRAIN_LIST = LIST_ROOT + 'train_gt.txt'
VAL_LIST   = LIST_ROOT + 'val_gt.txt'
TEST_LIST  = LIST_ROOT + 'test.txt'
TEST_SPLIT = LIST_ROOT + 'test_split/'

## 2. Helper Functions

### 2.1 Path Parsing

CULane list files have image and mask paths that require correction before use. `get_paths` handles this normalization.

In [ ]:
def get_paths(line, data_root):
    """
    Parse one line from a CULane list file and return absolute paths.

    Args:
        line (str): A single line from train_gt.txt / val_gt.txt.
        data_root (str): Absolute path to the dataset root.

    Returns:
        tuple[str, str]: (img_path, mask_path)
    """
    parts    = line.strip().split()
    img_rel  = parts[0].lstrip('/')   
    mask_rel = parts[1].lstrip('/')   

    # Fix image path — add extra folder level
    for folder in ['driver_23_30frame', 
                   'driver_161_90frame',
                   'driver_182_30frame',
                   'driver_37_30frame',
                   'driver_100_30frame',
                   'driver_193_90frame']:
        if img_rel.startswith(folder):
            img_rel = f'{folder}/{img_rel}'
            break

    # Fix mask path — build directly
    # mask_rel looks like: laneseg_label_w16/driver_XX/clip/frame.png
    # we need:             laneseg_label_w16/laneseg_label_w16/driver_XX/clip/frame.png
    mask_rel = mask_rel.replace(
               'laneseg_label_w16/', 
               'laneseg_label_w16/laneseg_label_w16/', 
               1)   # only replace FIRST occurrence

    img_path  = data_root + img_rel
    mask_path = data_root + mask_rel

    return img_path, mask_path

### 2.2 Preprocessing, Patch Sampling, and Feature Extraction

- `load_and_preprocess`: resizes images and masks to 256×512 and binarizes the mask
- `sample_patches`: draws balanced lane/background patch pairs from a single image
- `extract_features`: computes HOG (9 orientations, 8×8 cells, 2×2 blocks) plus mean and std intensity, producing a feature vector per patch

In [ ]:
def load_and_preprocess(img_path, mask_path):
    #Load an image and its lane mask, resize both to 256×512, and binarize the mask.
    img  = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    # Resize
    img  = cv2.resize(img,  (512, 256))
    mask = cv2.resize(mask, (512, 256),
           interpolation=cv2.INTER_NEAREST)
    
    # Convert mask to binary
    mask = (mask > 0).astype(np.uint8)
    
    return img, mask

def sample_patches(img, mask, n=500, patch_size=32):
    """
    Sample n balanced patch pairs (lane vs background) from a single image.

    Returns equal numbers of lane and background patches. Patches that
    would extend outside the image boundary are silently dropped, so the
    returned count may be slightly less than 2*n.

    Returns:
        tuple[list, list]: (patches, labels) where label 1 = lane, 0 = background.
    """
    half    = patch_size // 2
    patches = []
    labels  = []
    
    lane_pixels = np.argwhere(mask == 1)
    bg_pixels   = np.argwhere(mask == 0)
    
    if len(lane_pixels) == 0:
        return [], []
    
    n = min(len(lane_pixels), n)
    
    lane_samples = lane_pixels[
                   np.random.choice(len(lane_pixels),
                   n, replace=False)]
    bg_samples   = bg_pixels[
                   np.random.choice(len(bg_pixels),
                   n, replace=False)]
    
    # Assign labels alongside samples
    all_samples = np.concatenate([lane_samples,
                                  bg_samples])
    all_labels  = [1]*n + [0]*n
    
    for (y, x), label in zip(all_samples, all_labels):
        if y-half < 0 or y+half >= img.shape[0]: continue
        if x-half < 0 or x+half >= img.shape[1]: continue
        
        patch = img[y-half:y+half, x-half:x+half]
        patches.append(patch)
        labels.append(label)  # ← only add label if patch is valid
    
    return patches, labels

def extract_features(patch):
    """
    Compute HOG + mean/std intensity features for a single BGR patch.

    Returns a 1-D float array: [hog_features..., mean_intensity, std_intensity].
    """
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    
    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        feature_vector=True)
    
    mean_intensity = gray.mean()
    std_intensity  = gray.std()
    
    return np.concatenate([hog_features,
                           [mean_intensity,
                            std_intensity]])

## 3. Building the Training Set

Iterates over 1,000 training images, sampling 500 balanced patches per image and extracting HOG features. Results are saved to disk to avoid recomputation.

In [ ]:
#Build training set
#Processes N_IMAGES images, sampling 500 balanced patches each.
#Increase N_IMAGES for better coverage (slower); reduce for quick experiments.

X_train = []
y_train = []

# Read training list
with open(TRAIN_LIST, 'r') as f:
    lines = f.readlines()

# Start with 1000 images first
N_IMAGES = 1000

for i, line in enumerate(lines[:N_IMAGES]):
    img_path, mask_path = get_paths(line, DATA_ROOT)
    
    # Skip if file does not exist
    if not os.path.exists(img_path):  continue
    if not os.path.exists(mask_path): continue
    
    img, mask = load_and_preprocess(img_path, mask_path)
    patches, labels = sample_patches(img, mask, n=500)
    
    # Skip if no patches
    if len(patches) == 0: continue
    
    features = [extract_features(p) for p in patches]
    
    X_train.extend(features)
    y_train.extend(labels)
    
    # Print progress every 100 images
    if (i+1) % 100 == 0:
        print(f'Processed {i+1}/{N_IMAGES} images, '
              f'total patches: {len(X_train)}')

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f'\nDone!')
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'Lane patches: {y_train.sum()}')
print(f'Background patches: {(y_train==0).sum()}')

# Save to disk
np.save('/kaggle/working/X_train.npy', X_train)
np.save('/kaggle/working/y_train.npy', y_train)
print('Saved to disk')

## 4. Building the Validation Set

Same pipeline as training, capped at 500 images.

In [ ]:
# Build validation set (up to 500 images)
# Same pipeline as training — sample patches, extract HOG features.
X_val = []
y_val = []

with open(VAL_LIST, 'r') as f:
    val_lines = f.readlines()

print(f'Total validation images: {len(val_lines)}')

# Use all validation images or a subset
N_VAL = min(500, len(val_lines))

for i, line in enumerate(val_lines[:N_VAL]):
    img_path, mask_path = get_paths(line, DATA_ROOT)
    
    if not os.path.exists(img_path):  continue
    if not os.path.exists(mask_path): continue
    
    img, mask = load_and_preprocess(img_path, mask_path)
    patches, labels = sample_patches(img, mask, n=500)
    
    if len(patches) == 0: continue
    
    features = [extract_features(p) for p in patches]
    
    X_val.extend(features)
    y_val.extend(labels)
    
    if (i+1) % 100 == 0:
        print(f'Processed {i+1}/{N_VAL} images, '
              f'total patches: {len(X_val)}')

X_val = np.array(X_val)
y_val = np.array(y_val)

print(f'\nDone!')
print(f'X_val shape: {X_val.shape}')
print(f'y_val shape: {y_val.shape}')

np.save('/kaggle/working/X_val.npy', X_val)
np.save('/kaggle/working/y_val.npy', y_val)
print('Saved to disk')

## 5. Baseline Model — Unscaled Features

Load the saved features and train a Logistic Regression with default hyperparameters (C=1.0) on raw, unscaled HOG features as a baseline.

In [ ]:
#Unscaled F1: 0.71
# Load saved features
X_train = np.load('/kaggle/working/X_train.npy')
y_train = np.load('/kaggle/working/y_train.npy')
X_val   = np.load('/kaggle/working/X_val.npy')
y_val   = np.load('/kaggle/working/y_val.npy')

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_val shape:   {X_val.shape}')
print(f'y_val shape:   {y_val.shape}')

# Train with default C=1.0 first
lr = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    n_jobs=-1)

print('\nTraining Logistic Regression...')
lr.fit(X_train, y_train)
print('Training complete')

# Validate
y_pred = lr.predict(X_val)
f1     = f1_score(y_val, y_pred)
print(f'Validation F1: {f1:.4f}')

# Save model
joblib.dump(lr, '/kaggle/working/lr_C1.pkl')
print('Model saved')

## 6. Feature Scaling

HOG values and intensity statistics occupy very different numeric ranges. Applying `StandardScaler` (zero mean, unit variance) before training meaningfully improves convergence and validation F1.

In [ ]:
# Feature scaling
# HOG values and intensity stats live on very different scales.
# StandardScaler normalises each feature to zero mean / unit variance,
# which significantly helps lbfgs convergence.

# Scale features
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

# Save scaler
joblib.dump(scaler, '/kaggle/working/scaler.pkl')
print('Scaler saved')

# Retrain with scaled features
lr = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    n_jobs=-1)

print('Training Logistic Regression with scaled features...')
lr.fit(X_train_scaled, y_train)
print('Training complete')

# Validate
y_pred = lr.predict(X_val_scaled)
f1     = f1_score(y_val, y_pred)
print(f'Validation F1 (scaled): {f1:.4f}')

# Save model
joblib.dump(lr, '/kaggle/working/lr_scaled_C1.pkl')
print('Model saved')

## 7. Hyperparameter Tuning — Regularisation Strength C

`C` controls the inverse of L2 regularisation strength: lower C = stronger regularisation = simpler boundary. We sweep 8 values across several orders of magnitude to find the best validation F1.

In [ ]:
# Hyperparameter search: regularisation strength C
# Lower C → stronger L2 regularisation → simpler decision boundary.
# Results show C=0.0001 gives the best validation F1 (≈0.713).

C_values = [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10]
results  = {}

for C in C_values:
    print(f'Training with C={C}...')
    
    lr = LogisticRegression(
        C=C,
        max_iter=1000,
        solver='lbfgs',
        n_jobs=-1)
    
    lr.fit(X_train_scaled, y_train)
    
    y_pred     = lr.predict(X_val_scaled)
    f1         = f1_score(y_val, y_pred)
    results[C] = f1
    
    print(f'C={C}: Validation F1 = {f1:.4f}')
    joblib.dump(lr, 
    f'/kaggle/working/lr_scaled_C{C}.pkl')

print('\n--- Summary ---')
for C, f1 in results.items():
    print(f'C={C}: F1 = {f1:.4f}')

best_C = max(results, key=results.get)
print(f'\nBest C: {best_C}')
print(f'Best Validation F1: {results[best_C]:.4f}')

### 7.1 Final Model: Best C = 0.0001

Retrain using C=0.0001 (best from sweep) and save as the production model.

In [ ]:
print('Retraining best model with C=0.0001...')

best_lr = LogisticRegression(
    C=0.0001,
    max_iter=1000,
    solver='lbfgs',
    n_jobs=-1)

best_lr.fit(X_train_scaled, y_train)
print('Training complete')

# Validate one more time to confirm
y_pred = best_lr.predict(X_val_scaled)
f1     = f1_score(y_val, y_pred)
print(f'Validation F1: {f1:.4f}')

# Save best model and scaler
joblib.dump(best_lr, '/kaggle/working/lr_best.pkl')
joblib.dump(scaler,  '/kaggle/working/scaler.pkl')
print('Best model and scaler saved')

## 8. Test Evaluation

Play the next section to load pre-trained models and skip sections 3–7.



In [ ]:
# Load pre-trained model
best_lr = joblib.load('/kaggle/input/datasets/nishantsuresh1/lr-hog/lr_best.pkl')
scaler  = joblib.load('/kaggle/input/datasets/nishantsuresh1/lr-hog/scaler.pkl')

### 8.1 Test Feature Extraction

CULane's test split lists contain only image paths (no mask column). `extract_test_features` infers mask paths from the test label directory and applies the same HOG pipeline used for training.

In [ ]:


def extract_test_features(list_path, data_root, 
                           n_images=200):
    """
       Extract HOG features from a test split list file.
       Test lists contain only image paths (no mask column), so
       mask paths are inferred by replacing .jpg with .png and
       using the test label directory.

       Returns:
           tuple[np.ndarray, np.ndarray]: (X_test, y_test)
    """
    X_test = []
    y_test = []
    
    with open(list_path, 'r') as f:
        lines = f.readlines()
    
    n_images = min(n_images, len(lines))
    
    for i, line in enumerate(lines[:n_images]):
        img_rel  = line.strip()
        mask_rel = img_rel.replace('.jpg', '.png')
        
        # Fix image path — add extra folder level
        for folder in ['driver_23_30frame',
                       'driver_161_90frame',
                       'driver_182_30frame',
                       'driver_37_30frame',
                       'driver_100_30frame',
                       'driver_193_90frame']:
            if img_rel.startswith(folder):
                img_rel = f'{folder}/{img_rel}'
                break
        
        # Build paths
        img_path  = data_root + img_rel
        mask_path = data_root + \
                    'laneseg_label_w16_test/' \
                    'laneseg_label_w16_test/' + mask_rel
        
        if not os.path.exists(img_path):  continue
        if not os.path.exists(mask_path): continue
        
        img, mask = load_and_preprocess(img_path,
                                         mask_path)
        patches, labels = sample_patches(img, mask,
                                          n=200)
        
        if len(patches) == 0: continue
        
        features = [extract_features(p)
                    for p in patches]
        X_test.extend(features)
        y_test.extend(labels)
    
    return np.array(X_test), np.array(y_test)

### 8.2 Per-Scenario Evaluation

CULane provides 9 test scenarios covering different real-world conditions. We evaluate F1 and IoU (Jaccard score) on each using 200 images per scenario.

In [ ]:
scenarios = {
    'Normal'    : 'test0_normal.txt',
    'Crowded'   : 'test1_crowd.txt',
    'Highlight' : 'test2_hlight.txt',
    'Shadow'    : 'test3_shadow.txt',
    'No Line'   : 'test4_noline.txt',
    'Arrow'     : 'test5_arrow.txt',
    'Curve'     : 'test6_curve.txt',
    'Cross'     : 'test7_cross.txt',
    'Night'     : 'test8_night.txt'
}

print(f'{"Scenario":<12} {"F1":>8} {"IoU":>8}')
print('-' * 30)

all_results = {}

for scenario, filename in scenarios.items():
    list_path = TEST_SPLIT + filename
    
    X_test, y_test = extract_test_features(
                     list_path, DATA_ROOT,
                     n_images=200)
    
    if len(X_test) == 0:
        print(f'{scenario:<12} No data found')
        continue
    
    # Scale using training scaler
    X_test_scaled = scaler.transform(X_test)
    
    # Predict
    y_pred = best_lr.predict(X_test_scaled)
    
    # Compute metrics
    f1  = f1_score(y_test, y_pred)
    iou = jaccard_score(y_test, y_pred)
    
    all_results[scenario] = {'F1': f1, 'IoU': iou}
    print(f'{scenario:<12} {f1:>8.4f} {iou:>8.4f}')

print('\nDone!')

### 8.3 Cross Scenario: False Positive Rate

The Cross scenario contains intersection footage with **no lane markings**, making F1/IoU undefined. Instead, we measure the False Positive Rate: what fraction of background patches the model incorrectly classifies as lane.

In [ ]:
# Special case: Cross scenario
# This scenario has no lane markings, so F1/IoU are undefined.
# Instead we measure the False Positive Rate: what fraction of background
# patches the model incorrectly labels as lane.

X_cross = []
y_cross = []

with open(TEST_SPLIT + 'test7_cross.txt', 'r') as f:
    cross_lines = f.readlines()

n_images = min(200, len(cross_lines))

for line in cross_lines[:n_images]:
    img_rel = line.strip()
    
    for folder in ['driver_23_30frame',
                   'driver_161_90frame',
                   'driver_182_30frame',
                   'driver_37_30frame',
                   'driver_100_30frame',
                   'driver_193_90frame']:
        if img_rel.startswith(folder):
            img_rel = f'{folder}/{img_rel}'
            break
    
    img_path = DATA_ROOT + img_rel
    
    if not os.path.exists(img_path): continue
    
    img = cv2.imread(img_path)
    img = cv2.resize(img, (512, 256))
    
    # Sample only background patches
    # since there are no lane pixels
    H, W    = img.shape[:2]
    half    = 16
    patches = []
    
    for _ in range(200):
        y = np.random.randint(half, H-half)
        x = np.random.randint(half, W-half)
        patch = img[y-half:y+half, x-half:x+half]
        patches.append(patch)
    
    features = [extract_features(p) for p in patches]
    X_cross.extend(features)
    y_cross.extend([0] * 200)  # all background

X_cross = np.array(X_cross)
y_cross = np.array(y_cross)

# Scale and predict
X_cross_scaled = scaler.transform(X_cross)
y_pred_cross   = best_lr.predict(X_cross_scaled)

# False positive rate — how many background pixels
# did the model incorrectly label as lane?
fp_rate = (y_pred_cross == 1).sum() / len(y_pred_cross)
print(f'Cross False Positive Rate: {fp_rate:.4f}')
print(f'This means {fp_rate*100:.2f}% of background '
      f'pixels were incorrectly labeled as lane')

### 8.4 Results Summary

Compile all scenario results into a DataFrame and save to CSV.

In [ ]:
# Add cross result to results dict
all_results['Cross'] = {
    'F1'  : None,
    'IoU' : None,
    'FPR' : fp_rate
}

# Save F1 and IoU results
df = pd.DataFrame({
    k: v for k, v in all_results.items() 
    if v['F1'] is not None
}, index=['F1', 'IoU']).T

print(df.to_string())
print(f'\nCross FPR: {fp_rate:.4f}')

df.to_csv('/kaggle/working/lr_results.csv')
print('\nSaved to disk')

## 9. Visualization

### 9.1 Sliding-Window Prediction

`predict_and_visualize` runs the model over every pixel position (step=2 for speed) and renders the original image, ground truth mask, and model prediction side by side.

In [ ]:
def predict_and_visualize(model, scaler, img_path, 
                           mask_path, title, 
                           patch_size=32):
    """
    Run a sliding-window prediction over one image and plot original /
    ground-truth / prediction side by side.

    Uses step=2 (skips every other pixel) for speed — dense prediction
    at step=1 would be ~4× slower with minimal visual difference.
    """
    
    # Load and preprocess
    img  = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    img_r  = cv2.resize(img,  (512, 256))
    mask_r = cv2.resize(mask, (512, 256),
             interpolation=cv2.INTER_NEAREST)
    mask_b = (mask_r > 0).astype(np.uint8)
    
    half      = patch_size // 2
    H, W      = img_r.shape[:2]
    pred_mask = np.zeros((H, W), dtype=np.uint8)
    
    # Slide over every pixel
    for y in range(half, H-half, 2):  # step=2 for speed
        for x in range(half, W-half, 2):
            patch    = img_r[y-half:y+half,
                             x-half:x+half]
            features = extract_features(patch)\
                       .reshape(1, -1)
            features_scaled = scaler.transform(features)
            pred     = model.predict(features_scaled)[0]
            pred_mask[y, x] = pred
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].imshow(cv2.cvtColor(img_r, 
                   cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'{title}\nOriginal Image')
    axes[0].axis('off')
    
    axes[1].imshow(mask_b, cmap='gray')
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')
    
    axes[2].imshow(pred_mask, cmap='gray')
    axes[2].set_title('LR Prediction')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/pred_{title}.png',
                dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved pred_{title}.png')

### 9.2 Qualitative Results

Generate visualizations for four representative scenarios: Normal, Night, Shadow, and Highlight — the conditions with the widest spread in model performance.

In [ ]:
# Pick one example from each key scenario
viz_scenarios = {
    'Normal'    : 'test0_normal.txt',
    'Night'     : 'test8_night.txt',
    'Shadow'    : 'test3_shadow.txt',
    'Highlight' : 'test2_hlight.txt'
}

for scenario, filename in viz_scenarios.items():
    # Get first image from each scenario
    with open(TEST_SPLIT + filename, 'r') as f:
        first_line = f.readline().strip()
    
    # Fix image path
    img_rel = first_line
    for folder in ['driver_23_30frame',
                   'driver_161_90frame',
                   'driver_182_30frame',
                   'driver_37_30frame',
                   'driver_100_30frame',
                   'driver_193_90frame']:
        if img_rel.startswith(folder):
            img_rel = f'{folder}/{img_rel}'
            break
    
    img_path  = DATA_ROOT + img_rel
    mask_path = DATA_ROOT + \
                'laneseg_label_w16_test/' \
                'laneseg_label_w16_test/' + \
                first_line.replace('.jpg', '.png')
    
    if not os.path.exists(img_path):
        print(f'{scenario}: Image not found')
        continue
    if not os.path.exists(mask_path):
        print(f'{scenario}: Mask not found')
        continue
    
    print(f'Generating visualization for {scenario}...')
    predict_and_visualize(best_lr, scaler,
                          img_path, mask_path,
                          scenario)